# 03 - Results analysis

Creates the comparison plots and confusion matrices used in the report.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

RESULTS_DIR = Path("/content/drive/MyDrive/dataset_cv/results_runs")
df = pd.read_csv(RESULTS_DIR / "results_summary.csv")
display(df.head())

In [ ]:
experiment_order = ["E1_all","E2_synth","E3_real","E4_balanced_no_mix"]

plt.figure(figsize=(8,5))
for model in df["model"].unique():
    subset = df[df["model"] == model]
    means = subset.groupby("experiment")["best_test_acc"].mean().reindex(experiment_order)
    plt.plot(means.index, means.values, marker="o", label=model)

plt.title("Accuracy vs scenariusz danych (E1-E4)")
plt.xlabel("Eksperyment")
plt.ylabel("Accuracy")
plt.ylim(0.6, 1.0)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_vs_experiment.png", dpi=200)
plt.show()

In [ ]:
aug_order = ["A0_none","A1_geom","A2_color","A3_mix"]

plt.figure(figsize=(8,5))
for model in df["model"].unique():
    subset = df[df["model"] == model]
    means = subset.groupby("augmentation")["best_test_acc"].mean().reindex(aug_order)
    plt.plot(means.index, means.values, marker="o", label=model)

plt.title("Accuracy vs wariant augmentacji")
plt.xlabel("Augmentacja")
plt.ylabel("Accuracy")
plt.ylim(0.6, 1.0)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_vs_augmentation.png", dpi=200)
plt.show()

In [ ]:
top10 = df.sort_values("best_test_acc", ascending=False).head(10)

plt.figure(figsize=(10,5))
plt.barh(top10["run_id"], top10["best_test_acc"])
plt.gca().invert_yaxis()
plt.xlabel("Accuracy")
plt.title("TOP 10 konfiguracji modeli")
plt.xlim(0.6, 1.0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "top10_models.png", dpi=200)
plt.show()

In [ ]:
best_mobile = df[df["model"]=="mobilenet_v2"].sort_values("best_test_acc", ascending=False).iloc[0]
best_resnet = df[df["model"]=="resnet18"].sort_values("best_test_acc", ascending=False).iloc[0]

class_names = sorted([
    p.name for p in Path("/content/drive/MyDrive/dataset_cv/train").iterdir()
    if p.is_dir()
])

def plot_confusion(run_row):
    run_dir = Path(run_row["model_path"]).parent
    cm = np.load(run_dir / "confusion_matrix.npy")
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix\\n{run_row['run_id']}")
    plt.tight_layout()
    plt.savefig(run_dir / "confusion_matrix.png", dpi=200)
    plt.show()

plot_confusion(best_mobile)
plot_confusion(best_resnet)